In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage,HumanMessage, BaseMessage
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
# from langsmith import traceable

In [20]:
load_dotenv()

True

In [21]:
llm = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

In [5]:
llm.invoke("Testing if pinging to llm. Only confirm yes.").content

'Yes'

In [22]:
### creating state
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [23]:
def chat_node(state:ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages':[response]}
    

In [24]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)


In [25]:
chatbot.get_graph().print_ascii()

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| chat_node |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [26]:
# initial_state = {'messages':[HumanMessage(content="What is the capital of India?")]}

In [27]:
# result = chatbot.invoke(initial_state)

In [12]:
result['messages'][-1].content

'The capital of India is **New Delhi**.'

In [30]:
print(result['messages'][1].content)

The capital of India is **New Delhi**.


In [28]:
thread_id = '1'
while True:
    response = []
    user_message = input("Type here:")
    print("User: ", user_message)
    if user_message.strip().lower() in ["bye","quit","exit"]:
        break
    else:
        config = {'configurable':{'thread_id':thread_id}}
        response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]}, config = config)
        print("AI: ", response['messages'][-1].content)
        

User:  Hi, is 1>0? yes or nor
AI:  Yes.
User:  bye


In [29]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi, is 1>0? yes or nor', additional_kwargs={}, response_metadata={}, id='9e27ca28-8023-454c-9b3b-b47a6d279bd2'), AIMessage(content='Yes.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019dbe3e-db13-7f00-8c34-b5d460e94f14-0', usage_metadata={'input_tokens': 12, 'output_tokens': 2, 'total_tokens': 91, 'input_token_details': {'cache_read': 0}})]}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13fa98-b9bd-6cc2-8001-ce334abd69ac'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-04-24T06:48:04.946656+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13fa98-acb7-6932-8000-1513e67252c2'}}, tasks=(), interrupts=())

In [51]:
for i in checkpointer.list(None):
    print(i)

CheckpointTuple(config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13fa98-b9bd-6cc2-8001-ce334abd69ac'}}, checkpoint={'v': 4, 'ts': '2026-04-24T06:48:04.946656+00:00', 'id': '1f13fa98-b9bd-6cc2-8001-ce334abd69ac', 'channel_versions': {'__start__': '00000000000000000000000000000002.0.5985003113993238', 'messages': '00000000000000000000000000000003.0.7916322823105012', 'branch:to:chat_node': '00000000000000000000000000000003.0.7916322823105012'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000001.0.1803509732946339'}, 'chat_node': {'branch:to:chat_node': '00000000000000000000000000000002.0.5985003113993238'}}, 'updated_channels': ['messages'], 'channel_values': {'messages': [HumanMessage(content='Hi, is 1>0? yes or nor', additional_kwargs={}, response_metadata={}, id='9e27ca28-8023-454c-9b3b-b47a6d279bd2'), AIMessage(content='Yes.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0,

In [40]:
for i in checkpointer.list(config=config):
    print(i.config)

{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13fa98-b9bd-6cc2-8001-ce334abd69ac'}}
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13fa98-acb7-6932-8000-1513e67252c2'}}
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13fa98-acb2-6bde-bfff-d3c9c87abd69'}}


In [50]:
for i in checkpointer.list(None):
    print(i.config['configurable']['thread_id'])

1
1
1


In [ ]:
thds = set()
for i in checkpointer.list(config=config):
    thds.add(i.config['configurable']['thread_id'])
    

In [61]:
print((thds[0]))

TypeError: 'set' object is not subscriptable

In [53]:
def retrieve_all_threads():
    all_threads = set()
    for checkpoint in checkpointer.list(None):
        config = checkpoint.config
        configurable = config.get('configurable', {})
        thread_id = configurable.get('thread_id')
        # checkpoint.config.get('configurable',{}).get('thread_id') ## Same as above
        if thread_id:
            return all_threads.add(thread_id)

In [54]:
retrieve_all_threads()